# 🇧🇴 Inflación Bolivia — Modelado con SARIMA

**Objetivo:** Analizar la serie de tiempo de inflación boliviana, identificar tendencia y estacionalidad, ajustar un modelo SARIMA y generar proyecciones con intervalos de confianza.

**Técnicas aplicadas:**
- Descomposición clásica de series de tiempo
- Test de estacionariedad (ADF + KPSS)
- Selección de parámetros ARIMA con AIC
- SARIMA — extensión estacional de ARIMA
- Diagnóstico de residuos
- Forecast con intervalos de confianza al 80% y 95%

**Fuente de datos:** Banco Mundial — Indicador FP.CPI.TOTL.ZG (Bolivia)  
**Período:** 2000-2024 (datos anuales) + IPC mensual disponible

> Este análisis complementa el [Sistema de Alertas Dólar Paralelo](https://github.com/Serius69/dolar-paralelo-bolivia-arima) — ambos aplican modelado de series de tiempo al contexto macroeconómico boliviano.

In [ ]:
# ─── DEPENDENCIAS ───────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats

from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox

import itertools
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['font.size'] = 11

print('✅ Librerías cargadas')

## 1. Carga y Contexto

In [ ]:
# ─── CARGA DE DATOS ───────────────────────────────────────────────────────────
# Opción A: Descargar automáticamente
# !python descargar_datos.py

# Opción B: Cargar CSV ya descargado
df = pd.read_csv('datos/inflacion_anual_bolivia.csv')
df['fecha'] = pd.to_datetime(df['anio'].astype(str) + '-01-01')
df = df.set_index('fecha').asfreq('AS')  # Annual Start
serie = df['inflacion_pct']

print(f'Período: {serie.index[0].year} - {serie.index[-1].year}')
print(f'Observaciones: {len(serie)}')
print(f'\nEstadísticos básicos:')
print(f'  Media:  {serie.mean():.2f}%')
print(f'  Máximo: {serie.max():.2f}% ({serie.idxmax().year})')
print(f'  Mínimo: {serie.min():.2f}% ({serie.idxmin().year})')
serie.tail(8)

In [ ]:
# ─── GRÁFICA: SERIE COMPLETA CON ANOTACIONES HISTÓRICAS ───────────────────────
fig, ax = plt.subplots(figsize=(14, 6))

ax.fill_between(serie.index, serie.values, alpha=0.15, color='#2E86C1')
ax.plot(serie.index, serie.values, color='#2E86C1', lw=2.5, marker='o', ms=5)
ax.axhline(serie.mean(), color='red', linestyle='--', lw=1.5,
           label=f'Media histórica: {serie.mean():.1f}%', alpha=0.7)
ax.axhline(0, color='black', lw=0.8, alpha=0.5)

# Anotaciones de eventos históricos relevantes
eventos = {
    2008: ('Boom commodity', 'arriba'),
    2020: ('COVID-19', 'abajo'),
    2022: ('Crisis reservas', 'arriba'),
}
for anio, (label, pos) in eventos.items():
    if anio in serie.index.year:
        val = serie[serie.index.year == anio].values[0]
        xytext = (0, 20) if pos == 'arriba' else (0, -30)
        ax.annotate(label, xy=(pd.Timestamp(f'{anio}-01-01'), val),
                    xytext=xytext, textcoords='offset points',
                    fontsize=9, color='#7F8C8D',
                    arrowprops=dict(arrowstyle='->', color='#7F8C8D', lw=1))

ax.set_title('Inflación Anual Bolivia (2000-2024)\nFuente: Banco Mundial — FP.CPI.TOTL.ZG',
             fontweight='bold')
ax.set_ylabel('Inflación (%)')
ax.legend()
plt.tight_layout()
plt.savefig('img/01_inflacion_historica.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Descomposición de la Serie

In [ ]:
# ─── DESCOMPOSICIÓN CLÁSICA ────────────────────────────────────────────────────
# Descompone la serie en: Tendencia + Estacionalidad + Residuo
# Para datos anuales, period=1 (no hay ciclo intraserie)
# Nota: la estacionalidad real requiere datos mensuales; aquí analizamos tendencia y ciclo

# Suavizado Hodrick-Prescott para tendencia
from statsmodels.tsa.filters.hp_filter import hpfilter

ciclo, tendencia = hpfilter(serie.dropna(), lamb=100)  # lamb=100 para datos anuales

fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)

axes[0].plot(serie.index, serie.values, lw=2, color='#2E86C1', label='Serie original')
axes[0].plot(tendencia.index, tendencia.values, lw=2.5, color='red', label='Tendencia (HP)')
axes[0].set_title('Serie Original y Tendencia (Hodrick-Prescott)', fontweight='bold')
axes[0].legend(); axes[0].set_ylabel('Inflación %')

axes[1].bar(ciclo.index, ciclo.values, color=np.where(ciclo.values >= 0, '#2E86C1', '#E74C3C'), alpha=0.7)
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set_title('Componente Cíclica (desviación respecto a tendencia)', fontweight='bold')
axes[1].set_ylabel('Ciclo (pp)')

# Primeras diferencias
diff1 = serie.diff().dropna()
axes[2].bar(diff1.index, diff1.values, color=np.where(diff1.values >= 0, '#27AE60', '#E74C3C'), alpha=0.7)
axes[2].axhline(0, color='black', lw=0.8)
axes[2].set_title('Primera Diferencia (Δinflación)', fontweight='bold')
axes[2].set_ylabel('Δ% pp')

plt.tight_layout()
plt.savefig('img/02_descomposicion.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Test de Estacionariedad

In [ ]:
# ─── TEST ADF Y KPSS ──────────────────────────────────────────────────────────
# ADF  → H0: serie tiene raíz unitaria (no estacionaria)
#         Si p < 0.05: rechazamos H0 → serie ES estacionaria
# KPSS → H0: serie es estacionaria
#         Si p < 0.05: rechazamos H0 → serie NO ES estacionaria

def test_estacionariedad(serie, nombre='Serie'):
    print(f'\n{'─'*50}')
    print(f'Test de Estacionariedad: {nombre}')
    print(f'{'─'*50}')

    # ADF
    adf_result = adfuller(serie.dropna(), autolag='AIC')
    adf_p = adf_result[1]
    print(f'ADF  | Estadístico: {adf_result[0]:.4f} | p-value: {adf_p:.4f}', end=' ')
    print('→ ESTACIONARIA ✅' if adf_p < 0.05 else '→ NO ESTACIONARIA ⚠️')

    # KPSS
    kpss_result = kpss(serie.dropna(), regression='c', nlags='auto')
    kpss_p = kpss_result[1]
    print(f'KPSS | Estadístico: {kpss_result[0]:.4f} | p-value: {kpss_p:.4f}', end=' ')
    print('→ ESTACIONARIA ✅' if kpss_p > 0.05 else '→ NO ESTACIONARIA ⚠️')

    # Interpretación conjunta
    if adf_p < 0.05 and kpss_p > 0.05:
        print('📌 Conclusión: AMBOS tests coinciden → Serie ESTACIONARIA (d=0)')
    elif adf_p >= 0.05 and kpss_p <= 0.05:
        print('📌 Conclusión: AMBOS tests coinciden → Serie NO ESTACIONARIA (aplicar diferenciación)')
    else:
        print('📌 Conclusión: Tests contradictorios → Evaluar con diferenciación')

    return adf_p, kpss_p

adf_p, kpss_p = test_estacionariedad(serie, 'Inflación original')
adf_p1, kpss_p1 = test_estacionariedad(serie.diff().dropna(), 'Primera diferencia (d=1)')

In [ ]:
# ─── ACF Y PACF ───────────────────────────────────────────────────────────────
serie_usar = serie.diff().dropna() if adf_p >= 0.05 else serie
titulo = 'Primera diferencia' if adf_p >= 0.05 else 'Serie original'

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_acf(serie_usar, lags=min(15, len(serie_usar)//2 - 1),
         ax=axes[0], title=f'ACF — {titulo}\n→ Identifica orden q (MA)')
plot_pacf(serie_usar, lags=min(15, len(serie_usar)//2 - 1),
          ax=axes[1], title=f'PACF — {titulo}\n→ Identifica orden p (AR)', method='ywm')
plt.tight_layout()
plt.savefig('img/03_acf_pacf.png', dpi=150, bbox_inches='tight')
plt.show()

print('💡 Interpretación:')
print('   ACF  decae lentamente → componente AR (p > 0)')
print('   PACF corte abrupto en lag k → p = k')
print('   ACF  corte abrupto en lag k → q = k')

## 4. Selección y Ajuste del Modelo SARIMA

In [ ]:
# ─── SELECCIÓN AUTOMÁTICA POR AIC ────────────────────────────────────────────
# Para datos anuales: sin componente estacional (S=1)
# Para datos mensuales usaríamos S=12

def seleccionar_arima(serie, max_p=3, max_d=2, max_q=3, verbose=True):
    """Grid search de órdenes ARIMA minimizando AIC."""
    resultados = []
    total = (max_p+1) * (max_d+1) * (max_q+1)
    print(f'Evaluando {total} combinaciones ARIMA(p,d,q)...')

    for p, d, q in itertools.product(range(max_p+1), range(max_d+1), range(max_q+1)):
        try:
            modelo = SARIMAX(serie, order=(p,d,q),
                             trend='c', enforce_stationarity=False,
                             enforce_invertibility=False)
            ajuste = modelo.fit(disp=False)
            resultados.append({'orden': (p,d,q), 'aic': ajuste.aic, 'bic': ajuste.bic})
        except:
            pass

    df_res = pd.DataFrame(resultados).sort_values('aic').reset_index(drop=True)
    if verbose:
        print('\nTop 5 modelos por AIC:')
        print(df_res.head().to_string(index=False))
    return df_res.iloc[0]['orden']

mejor_orden = seleccionar_arima(serie.dropna())
print(f'\n🏆 Mejor modelo: ARIMA{mejor_orden}')

In [ ]:
# ─── AJUSTAR MODELO FINAL ─────────────────────────────────────────────────────
modelo_final = SARIMAX(
    serie.dropna(),
    order=mejor_orden,
    trend='c',
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

print(modelo_final.summary().tables[0])
print(modelo_final.summary().tables[1])

## 5. Diagnóstico de Residuos

In [ ]:
# ─── DIAGNÓSTICO ─────────────────────────────────────────────────────────────
# Un buen modelo ARIMA debe tener residuos que sean:
#  ✅ Centrados en 0 (sin sesgo)
#  ✅ Sin autocorrelación (test Ljung-Box)
#  ✅ Aproximadamente normales

residuos = modelo_final.resid

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# 1. Residuos en el tiempo
axes[0,0].plot(residuos.index, residuos.values, color='#2E86C1', lw=1.5)
axes[0,0].axhline(0, color='red', linestyle='--', lw=1)
axes[0,0].set_title('Residuos en el tiempo\n(deben estar centrados en 0)', fontweight='bold')
axes[0,0].set_ylabel('Residuo')

# 2. Histograma + curva normal
residuos_norm = (residuos - residuos.mean()) / residuos.std()
axes[0,1].hist(residuos_norm, bins=15, density=True, alpha=0.7, color='#2E86C1', label='Residuos')
x = np.linspace(-4, 4, 100)
axes[0,1].plot(x, stats.norm.pdf(x), 'r-', lw=2, label='Normal teórica')
_, p_norm = stats.normaltest(residuos.dropna())
axes[0,1].set_title(f'Distribución de residuos\nTest normalidad p={p_norm:.3f}', fontweight='bold')
axes[0,1].legend()

# 3. QQ Plot
stats.probplot(residuos.dropna(), dist='norm', plot=axes[1,0])
axes[1,0].set_title('Q-Q Plot (normalidad de residuos)', fontweight='bold')

# 4. ACF de residuos (debe no tener autocorrelación)
plot_acf(residuos.dropna(), lags=min(10, len(residuos)//2 - 1),
         ax=axes[1,1], title='ACF de Residuos\n(barras dentro de bandas = sin autocorrelación ✅)')

plt.tight_layout()
plt.savefig('img/04_diagnostico_residuos.png', dpi=150, bbox_inches='tight')
plt.show()

# Test Ljung-Box
lb = acorr_ljungbox(residuos.dropna(), lags=[5], return_df=True)
print(f'\nTest Ljung-Box (autocorrelación en residuos):')
print(f'   p-value: {lb["lb_pvalue"].values[0]:.4f}', end=' ')
print('→ Sin autocorrelación ✅' if lb['lb_pvalue'].values[0] > 0.05 else '→ Hay autocorrelación ⚠️')

## 6. Forecast — Proyección con Intervalos de Confianza

In [ ]:
# ─── FORECAST ─────────────────────────────────────────────────────────────────
HORIZONTE_AÑOS = 4  # Proyectar 4 años hacia adelante

forecast = modelo_final.get_forecast(steps=HORIZONTE_AÑOS)
fc_mean  = forecast.predicted_mean
fc_ci80  = forecast.conf_int(alpha=0.20)  # IC al 80%
fc_ci95  = forecast.conf_int(alpha=0.05)  # IC al 95%

# Fechas de proyección
ultimo_anio = serie.dropna().index[-1].year
fechas_fc = pd.date_range(start=f'{ultimo_anio+1}-01-01', periods=HORIZONTE_AÑOS, freq='AS')
fc_mean.index = fechas_fc
fc_ci80.index = fechas_fc
fc_ci95.index = fechas_fc

# ─── GRÁFICA FINAL ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 7))

# Serie histórica
ax.plot(serie.index, serie.values, color='#2E86C1', lw=2.5, label='Inflación histórica', zorder=3)
ax.scatter(serie.index, serie.values, color='#2E86C1', s=30, zorder=4)

# Valores ajustados (in-sample)
fitted = modelo_final.fittedvalues
ax.plot(fitted.index, fitted.values, color='#E67E22', lw=1.5, linestyle='--',
        label='Valores ajustados (in-sample)', alpha=0.8)

# Forecast
ax.plot(fc_mean.index, fc_mean.values, color='#E74C3C', lw=2.5,
        marker='D', ms=7, label=f'Proyección {ultimo_anio+1}-{ultimo_anio+HORIZONTE_AÑOS}', zorder=3)

# Intervalos de confianza
ax.fill_between(fc_ci95.index, fc_ci95.iloc[:,0], fc_ci95.iloc[:,1],
                alpha=0.15, color='#E74C3C', label='IC 95%')
ax.fill_between(fc_ci80.index, fc_ci80.iloc[:,0], fc_ci80.iloc[:,1],
                alpha=0.30, color='#E74C3C', label='IC 80%')

# Línea vertical separando histórico de forecast
ax.axvline(pd.Timestamp(f'{ultimo_anio}-01-01'), color='gray', linestyle=':',
           lw=2, label='Inicio proyección')
ax.axhline(0, color='black', lw=0.8, alpha=0.4)

ax.set_title(f'Inflación Bolivia — Proyección {ultimo_anio+1}-{ultimo_anio+HORIZONTE_AÑOS}\n'
             f'Modelo ARIMA{mejor_orden} | Fuente: Banco Mundial',
             fontweight='bold', fontsize=13)
ax.set_ylabel('Inflación anual (%)')
ax.legend(loc='upper left', fontsize=9)
ax.set_ylim(bottom=min(serie.min()-2, fc_ci95.iloc[:,0].min()-1))

plt.tight_layout()
plt.savefig('img/05_forecast_inflacion.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📊 Proyecciones puntuales:')
for fecha, val in zip(fc_mean.index, fc_mean.values):
    ci95_lo = fc_ci95.iloc[list(fc_mean.index).index(fecha), 0]
    ci95_hi = fc_ci95.iloc[list(fc_mean.index).index(fecha), 1]
    print(f'   {fecha.year}: {val:.2f}%  [IC95%: {ci95_lo:.2f}% — {ci95_hi:.2f}%]')

## 7. Conclusiones

In [ ]:
# ─── RESUMEN ─────────────────────────────────────────────────────────────────
print('=' * 60)
print('RESUMEN — MODELADO INFLACIÓN BOLIVIA')
print('=' * 60)
print(f'\n📅 Período histórico analizado: {serie.index[0].year}-{serie.index[-1].year}')
print(f'📐 Modelo seleccionado: ARIMA{mejor_orden} (minimizando AIC)')
print(f'🔬 Diagnóstico residuos: p-value Ljung-Box = {lb["lb_pvalue"].values[0]:.4f}')

print(f'\n📊 Estadísticos históricos:')
print(f'   Inflación media: {serie.mean():.2f}%')
print(f'   Máximo: {serie.max():.2f}% ({serie.idxmax().year})')
print(f'   Mínimo: {serie.min():.2f}% ({serie.idxmin().year})')

print(f'\n🔮 Proyecciones (punto central):')
for fecha, val in zip(fc_mean.index, fc_mean.values):
    print(f'   {fecha.year}: {val:.2f}%')

print(f'\n💡 Limitaciones del modelo:')
print(f'   - Datos anuales: baja frecuencia → IC amplios en forecast')
print(f'   - ARIMA no captura shocks exógenos (crisis política, commodity prices)')
print(f'   - Para mayor precisión: usar datos mensuales del IPC (INE Bolivia)')
print(f'\n🔗 Complementar con: análisis del dólar paralelo (ver dolar-paralelo-bolivia-arima)')